[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fedgutcor/lab-ia-intermedio/blob/main/encuentros/e03_texto_a_numeros.ipynb)

# Encuentro 3 · Cómo se convierte el texto en números

**Laboratorio de IA · nivel intermedio · Semana 2 · primera clase práctica**

Propósito: ver cómo la preparación y la forma de representar el texto cambian lo que responde un modelo, y elegir un modelo que convenga a tu tarea.

Al terminar tienes tres cosas:

1. **Tu proyecto elegido**, con un resumen corto: la tarea, quién usa la predicción, el error que más importa y cómo vas a medirlo.
2. **Una primera predicción** de un modelo que ya existe y de otro que entrenas tú en segundos.
3. **Una representación documentada** y un caso difícil en el que viste su efecto.

| Parte | Qué haces |
|---|---|
| 1 | Cargar los datos y partirlos una sola vez |
| 2 | Ver cómo tres herramientas parten el mismo texto |
| 3 | Comprobar qué borra una «limpieza» muy común |
| 4 | Comparar cinco representaciones con la misma prueba |
| 5 | Usar un modelo preentrenado después de leer su ficha |
| 6 | Mirar los mismos cuatro casos en todos los modelos |
| 7 | **Tu proyecto**: elegir, resumir y obtener tu primera predicción |

**Cómo trabajar:** cuando una parte diga **Predice**, escribe tu respuesta antes de ejecutar. Después ejecuta y compara lo que esperabas con lo que salió.

Para ejecutar todo: *Entorno de ejecución → Ejecutar todo*. No hace falta tarjeta gráfica.

## 0 · Entorno

Colab ya trae todo lo que usa este notebook. Si trabajas en otro entorno y falta algo, quita el comentario de la línea `!pip`.

In [ ]:
# Solo fuera de Colab, si falta algo:
# !pip install -q tensorflow spacy transformers torch sentencepiece scikit-learn pandas

import json, os, platform, datetime, time, re
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import GroupShuffleSplit, train_test_split

def como_texto(x):
    # Keras 3 recibe el texto como tensor de TensorFlow, no como lista ni como columna de pandas
    return tf.constant([str(t) for t in x])

keras.utils.set_random_seed(0)
INICIO = time.time()
ENTORNO = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "maquina": os.environ.get("LAB_MAQUINA", platform.platform()),
    "fecha": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
}
SALIDAS = {"encuentro": 3, "entorno": ENTORNO}
ENTORNO

## 1 · Los datos y una sola partición

Son 900 mensajes de soporte **ficticios**, cada uno con su motivo correcto: *acceso a la cuenta*, *devolución*, *envío* o *facturación*. La columna `dificultad` marca los mensajes con **negación** («no quiero devolver nada…») y los **ambiguos**.

Los datos se parten **una sola vez** y todos los modelos se evalúan con la misma prueba. Se parten las **frases**, no los mensajes: los mensajes que dicen lo mismo con otras palabras quedan del mismo lado, así la prueba trae maneras de decir las cosas que ningún modelo vio al entrenar.

Además se pide que la prueba tenga frases de cada tipo de dificultad. Hay pocas frases con negación: una partición al azar puede dejar en la prueba apenas 5 mensajes con negación, y con eso no se puede comparar nada. En el Encuentro 4 verás qué pasa si se parte mal.

In [ ]:
DATOS = "https://raw.githubusercontent.com/Fedgutcor/lab-ia-intermedio/main/datos/"
df = pd.read_csv(DATOS + "mensajes_soporte.csv")
MOTIVOS = sorted(df["motivo"].unique())
df["y"] = df["motivo"].map(MOTIVOS.index)

# se parten las FRASES (no los mensajes), con frases de cada dificultad en la prueba
frases = df.groupby("frase")["dificultad"].first()
f_ent, f_pru = train_test_split(frases.index, test_size=0.3, stratify=frases.values, random_state=0)
ent = df[df["frase"].isin(f_ent)].reset_index(drop=True)
pru = df[df["frase"].isin(f_pru)].reset_index(drop=True)

print("Motivos:", MOTIVOS)
print(f"Entrenamiento: {len(ent)} mensajes · Prueba: {len(pru)} mensajes")
print("Prueba por dificultad:", pru["dificultad"].value_counts().to_dict())
SALIDAS["datos"] = {"total": len(df), "entrenamiento": len(ent), "prueba": len(pru), "motivos": MOTIVOS,
                    "prueba_por_dificultad": {k: int(v) for k, v in pru["dificultad"].value_counts().items()},
                    "frases_prueba": int(pru["frase"].nunique()),
                    "frases_por_dificultad": {k: int(v) for k, v in pru.groupby("dificultad")["frase"].nunique().items()}}
pru[["mensaje", "motivo", "dificultad"]].head(6)

## 2 · El mismo texto, tres maneras de partirlo

Un modelo no lee letras: lee números. Antes hay que **tokenizar**, es decir, partir el texto en pedazos. Cada herramienta lo hace a su modo y para un fin distinto:

| Herramienta | Qué hace | Para qué sirve |
|---|---|---|
| **spaCy** | Parte en palabras y signos, con reglas del idioma | Preparar y analizar texto: oraciones, palabras vacías, lemas |
| **Keras `TextVectorization`** | Arma un vocabulario con las palabras del entrenamiento y cambia cada una por su número | Alimentar un modelo que entrenas tú |
| **Tokenizador de Transformers** | Parte en *subpalabras* con un vocabulario fijo | Alimentar un modelo preentrenado (parte 5) |

Una palabra que no está en el vocabulario de Keras se vuelve `[UNK]` (desconocida) y recibe el número 1.

Estos son los cuatro casos de hoy, escritos para la clase. **No están en los datos.**

**Predice:** en el caso C, ¿qué palabras crees que el vocabulario de Keras no conoce?

In [ ]:
import spacy
CASOS = pd.DataFrame([
    {"id": "A", "mensaje": "Quiero devolver la chaqueta porque me quedó pequeña.", "referencia": "devolución"},
    {"id": "B", "mensaje": "No quiero devolver la chaqueta, solo saber cuándo llega.", "referencia": "envío"},
    {"id": "C", "mensaje": "Me sacaron la plata dos veces por el mismo pedido.", "referencia": "facturación"},
    {"id": "D", "mensaje": "Buenas tardes, espero que estén muy bien, les cuento que desde ayer no puedo entrar a mi cuenta.",
     "referencia": "acceso a la cuenta"},
])
nlp = spacy.blank("es")          # reglas del español, sin descargar ningún modelo

palabras_a_numeros = layers.TextVectorization(max_tokens=5000)
palabras_a_numeros.adapt(como_texto(ent["mensaje"]))
VOCAB = palabras_a_numeros.get_vocabulary()
print("Palabras en el vocabulario de Keras:", len(VOCAB))
print("Las 15 más frecuentes:", VOCAB[2:17], "\n")

SALIDAS["tokens"] = {"vocabulario": len(VOCAB), "frecuentes": VOCAB[2:17], "casos": []}
for _, c in CASOS.iterrows():
    sp = [t.text for t in nlp(c["mensaje"])]
    ids = [int(i) for i in palabras_a_numeros(tf.constant([c["mensaje"]]))[0].numpy()]
    vistas = [VOCAB[i] for i in ids]
    desconocidas = [w for w in re.findall(r"\w+", c["mensaje"].lower()) if w not in set(VOCAB)]
    print(f'{c["id"]} · {c["mensaje"]}')
    print("   spaCy:", sp)
    print("   Keras:", vistas)
    print("   números:", ids)
    print("   desconocidas para Keras:", desconocidas or "ninguna", "\n")
    SALIDAS["tokens"]["casos"].append({"id": c["id"], "mensaje": c["mensaje"], "spacy": sp, "keras": vistas,
                                       "ids": ids, "desconocidas": desconocidas})

## 3 · Una «limpieza» que borra significado

Muchos tutoriales recomiendan quitar las **palabras vacías** (artículos, preposiciones, pronombres…) antes de entrenar. La lista de spaCy para español es la que usa mucha gente. Mira qué palabras trae.

**Predice:** ¿cómo quedan los casos A y B después de la limpieza? ¿Se siguen distinguiendo?

In [ ]:
from spacy.lang.es.stop_words import STOP_WORDS
PALABRAS_VACIAS = set(STOP_WORDS)        # Cambia esto: prueba quitando "no" de la lista → PALABRAS_VACIAS - {"no"}

def limpiar(texto):
    return " ".join(t.text.lower() for t in nlp(texto) if t.is_alpha and t.text.lower() not in PALABRAS_VACIAS)

print(f"La lista trae {len(PALABRAS_VACIAS)} palabras. ¿Incluye «no»?", "no" in PALABRAS_VACIAS)
print("Algunas que cambian el sentido:", sorted(PALABRAS_VACIAS & {"no", "nada", "nunca", "sin", "ni", "solo", "cuándo", "dónde"}), "\n")
SALIDAS["limpieza"] = {"total": len(PALABRAS_VACIAS), "incluye_no": "no" in PALABRAS_VACIAS,
                       "sentido": sorted(PALABRAS_VACIAS & {"no", "nada", "nunca", "sin", "ni", "solo", "cuándo", "dónde"}), "casos": []}
for _, c in CASOS.iterrows():
    print(f'{c["id"]} · {c["mensaje"]}\n    → {limpiar(c["mensaje"])}')
    SALIDAS["limpieza"]["casos"].append({"id": c["id"], "antes": c["mensaje"], "despues": limpiar(c["mensaje"])})

## 4 · Cinco representaciones, una sola prueba

| Modelo | Qué ve del mensaje |
|---|---|
| Línea base | Nada: responde siempre el motivo más frecuente del entrenamiento |
| Bolsa de palabras | Cuántas veces aparece cada palabra, sin importar el orden |
| Bolsa con pares | Lo anterior más los pares de palabras seguidas («no quiero», «quiero devolver») |
| Bolsa sin palabras vacías | La bolsa de palabras, después de la limpieza de la parte 3 |
| Secuencia con *embedding* | Las palabras en orden; cada una se vuelve un vector de 16 números que el modelo aprende |
| Secuencia cortada en 6 | Igual que la anterior, pero solo con las primeras 6 palabras |

La línea base no es relleno: es la vara para saber si los demás aprendieron algo.

**Predice:** ¿cuál acierta más en toda la prueba? ¿Y en los mensajes con negación?

Tarda unos segundos por modelo.

In [ ]:
def entrenar(textos, y, vectorizador, secuencia=False, epocas=30, clases=None):
    vectorizador.adapt(como_texto(textos))
    capas = [keras.Input(shape=(1,), dtype="string"), vectorizador]
    if secuencia:
        capas += [layers.Embedding(vectorizador.vocabulary_size(), 16), layers.GlobalAveragePooling1D()]
    capas += [layers.Dense(clases or len(MOTIVOS), activation="softmax")]
    modelo = keras.Sequential(capas)
    modelo.compile(optimizer=keras.optimizers.Adam(0.01), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    modelo.fit(como_texto(textos), np.asarray(y), epochs=epocas, batch_size=32, verbose=0)
    return modelo

def representaciones():
    # (nombre, vectorizador, ¿secuencia?, preparación previa)
    return [("Bolsa de palabras", layers.TextVectorization(max_tokens=5000, output_mode="count"), False, None),
            ("Bolsa con pares", layers.TextVectorization(max_tokens=10000, ngrams=2, output_mode="count"), False, None),
            ("Bolsa sin palabras vacías", layers.TextVectorization(max_tokens=5000, output_mode="count"), False, limpiar),
            ("Secuencia con embedding", layers.TextVectorization(max_tokens=5000, output_sequence_length=40), True, None),
            ("Secuencia cortada en 6", layers.TextVectorization(max_tokens=5000, output_sequence_length=6), True, None)]

def entrenar_todos(datos):
    modelos, prepara = {}, {}
    for nombre, vectorizador, secuencia, preparar in representaciones():
        textos = datos["mensaje"].map(preparar) if preparar else datos["mensaje"]
        modelos[nombre] = entrenar(textos, datos["y"], vectorizador, secuencia)
        if preparar:
            prepara[nombre] = preparar      # la misma preparación tiene que aplicarse al predecir
    return modelos, prepara

def probabilidades(nombre, textos, modelos=None, prepara=None):
    modelos = modelos or MODELOS
    preparar = (PREPARA if prepara is None else prepara).get(nombre, lambda t: t)
    return modelos[nombre].predict(como_texto([preparar(t) for t in textos]), verbose=0)

MODELOS, PREPARA = entrenar_todos(ent)
BASE = int(ent["y"].mode()[0])
PRED = {"Línea base": np.full(len(pru), BASE)}
for nombre in MODELOS:
    PRED[nombre] = probabilidades(nombre, pru["mensaje"]).argmax(axis=1)

real, neg = pru["y"].values, (pru["dificultad"] == "negación").values
RESULTADOS = pd.DataFrame({n: {"Todos": (p == real).mean(), "Con negación": (p[neg] == real[neg]).mean()}
                           for n, p in PRED.items()}).T.round(3)
sub = pru[neg]
print(f"Prueba: {len(pru)} mensajes de {pru['frase'].nunique()} frases.")
print(f"Con negación: {len(sub)} mensajes de {sub['frase'].nunique()} frases · motivos: {sub['motivo'].value_counts().to_dict()}")
print(f"La línea base responde siempre «{MOTIVOS[BASE]}».")
SALIDAS["resultados"] = {"base_motivo": MOTIVOS[BASE],
                         "negacion": {"n": len(sub), "frases": int(sub["frase"].nunique()),
                                      "motivos": {k: int(v) for k, v in sub["motivo"].value_counts().items()}},
                         "filas": [{"modelo": n, **{k: float(v) for k, v in f.items()}} for n, f in RESULTADOS.iterrows()]}
RESULTADOS

**Antes de sacar conclusiones, mira los conteos.** Si la línea base, que ni siquiera lee el mensaje, acierta todos los mensajes con negación, no es porque entienda la negación: es porque en esta prueba todos esos mensajes tienen el motivo que ella responde siempre. Con tan pocos mensajes y tan pocas frases, esa columna sirve para mirar casos, no para elegir un modelo.

### Lo que se pierde al cortar

La secuencia cortada en 6 solo ve el principio del mensaje. Si el motivo está al final, no llega al modelo. Esta celda cuenta cuántos mensajes de la prueba se cortan y muestra uno afectado.

In [ ]:
largo = pru["mensaje"].map(lambda t: len(re.findall(r"\w+", t.lower())))
cortados = int((largo > 6).sum())
print(f"{cortados} de {len(pru)} mensajes de prueba tienen más de 6 palabras: el modelo cortado no ve el resto.\n")

completo, corto = PRED["Secuencia con embedding"], PRED["Secuencia cortada en 6"]
afectados = pru[(completo == pru["y"].values) & (corto != pru["y"].values)]
vec6 = MODELOS["Secuencia cortada en 6"].layers[0]
SALIDAS["truncado"] = {"largo": 6, "cortados": cortados, "prueba": len(pru), "afectados": len(afectados), "ejemplo": None}
if len(afectados):
    m = afectados.iloc[0]
    ids = [int(i) for i in vec6(tf.constant([m["mensaje"]]))[0].numpy()]
    visto = [vec6.get_vocabulary()[i] for i in ids]
    print("Mensaje completo:", m["mensaje"])
    print("Lo que ve el modelo cortado:", visto)
    print("Motivo correcto:", m["motivo"], "· secuencia completa:", MOTIVOS[completo[afectados.index[0]]],
          "· cortada en 6:", MOTIVOS[corto[afectados.index[0]]])
    SALIDAS["truncado"]["ejemplo"] = {"mensaje": m["mensaje"], "visto": visto, "referencia": m["motivo"],
                                      "completa": MOTIVOS[completo[afectados.index[0]]], "cortada": MOTIVOS[corto[afectados.index[0]]]}
print(f"\nMensajes que la secuencia completa acierta y la cortada falla: {len(afectados)}")

## 4b · ¿Ganaría el mismo con otra partición?

La prueba tiene pocas frases distintas. Si cambian las frases que caen en la prueba, ¿cambia el ganador? Esta celda repite todo con 5 particiones distintas. Tarda menos de un minuto.

**Predice:** ¿gana el mismo modelo las 5 veces?

In [ ]:
filas = []
for semilla in range(5):
    keras.utils.set_random_seed(0)
    fe, fp = train_test_split(frases.index, test_size=0.3, stratify=frases.values, random_state=semilla)
    e_ = df[df["frase"].isin(fe)].reset_index(drop=True)
    p_ = df[df["frase"].isin(fp)].reset_index(drop=True)
    mods, preps = entrenar_todos(e_)
    fila = {"partición": semilla, "Línea base": float((p_["y"] == e_["y"].mode()[0]).mean())}
    for nombre in mods:
        fila[nombre] = float((probabilidades(nombre, p_["mensaje"], mods, preps).argmax(axis=1) == p_["y"].values).mean())
    filas.append(fila)
PARTICIONES = pd.DataFrame(filas).set_index("partición").round(3)
PARTICIONES["Gana"] = PARTICIONES.drop(columns="Línea base").idxmax(axis=1)
promedio = PARTICIONES.drop(columns="Gana").mean().round(3)
SALIDAS["particiones"] = {"filas": [{"particion": int(i), **f} for i, f in PARTICIONES.iterrows()],
                          "promedio": {k: float(v) for k, v in promedio.items()}}
print("Promedio de las 5 particiones:")
print(promedio.to_string(), "\n")
PARTICIONES

## 5 · Un modelo que ya existe, después de leer su ficha

Antes de usar un modelo preentrenado conviene leer su **ficha** (*model card*): en qué idiomas se entrenó, para qué tarea, con qué licencia y cuánto texto acepta. Es el mismo modelo del Encuentro 1. No conoce nuestros motivos, pero puede comparar cada motivo con el mensaje (*zero-shot*).

Su tokenizador trabaja con **subpalabras**: una palabra rara se parte en pedazos conocidos, así que casi nunca aparece un `[UNK]`.

La primera vez descarga el modelo: unos 550 MB, una sola vez por sesión de Colab.

In [ ]:
import transformers, torch
from huggingface_hub import ModelCard
from transformers import AutoTokenizer, AutoConfig, pipeline
ENTORNO["transformers"] = transformers.__version__
ENTORNO["torch"] = torch.__version__
ENTORNO["spacy"] = spacy.__version__

MODELO = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
REVISION = "8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c"   # versión fija: el resultado no cambia si el autor actualiza el modelo

tarjeta = ModelCard.load(MODELO)          # la ficha publicada por el autor
cfg = AutoConfig.from_pretrained(MODELO, revision=REVISION)
tok = AutoTokenizer.from_pretrained(MODELO, revision=REVISION)
idiomas = tarjeta.data.language if isinstance(tarjeta.data.language, list) else [tarjeta.data.language]
FICHA = {"modelo": MODELO, "revision": REVISION, "tarea": tarjeta.data.pipeline_tag, "licencia": tarjeta.data.license,
         "idiomas": idiomas, "incluye_espanol": "es" in idiomas, "max_tokens": cfg.max_position_embeddings,
         "vocabulario": tok.vocab_size}
for k, v in FICHA.items():
    print(f"{k:>16}: {v}")

print()
SALIDAS["subpalabras"] = []
for _, c in CASOS.iterrows():
    piezas = tok.tokenize(c["mensaje"])
    print(f'{c["id"]} · {len(piezas)} subpalabras: {piezas}')
    SALIDAS["subpalabras"].append({"id": c["id"], "piezas": piezas})
SALIDAS["ficha"] = FICHA

Ahora el modelo predice. Para no esperar demasiado en Colab se usa una muestra de la prueba: hasta 12 mensajes con negación y 12 normales. La bolsa de palabras responde sobre la **misma muestra**, para comparar con justicia.

**Predice:** ¿el modelo preentrenado, que nunca vio estos datos, acierta más o menos que la bolsa de palabras en los mensajes con negación?

In [ ]:
clasificador = pipeline("zero-shot-classification", model=MODELO, revision=REVISION, device=-1)
PLANTILLA = "Este mensaje trata sobre {}."

def zero_shot(textos, etiquetas):
    salidas = [clasificador(t, candidate_labels=list(etiquetas), hypothesis_template=PLANTILLA) for t in textos]
    return [dict(zip(s["labels"], [float(x) for x in s["scores"]])) for s in salidas]

neg, nor = pru[pru["dificultad"] == "negación"], pru[pru["dificultad"] == "normal"]
muestra = pd.concat([neg.sample(min(12, len(neg)), random_state=0), nor.sample(12, random_state=0)])
zs = zero_shot(muestra["mensaje"], MOTIVOS)
muestra = muestra.assign(preentrenado=[max(p, key=p.get) for p in zs],
                         bolsa=[MOTIVOS[i] for i in PRED["Bolsa de palabras"][muestra.index]])
resumen = {}
for d in ["negación", "normal"]:
    s = muestra[muestra["dificultad"] == d]
    resumen[d] = {"n": len(s), "preentrenado": int((s["preentrenado"] == s["motivo"]).sum()),
                  "bolsa": int((s["bolsa"] == s["motivo"]).sum())}
print(pd.DataFrame(resumen).T, "\n")
SALIDAS["muestra"] = {"resumen": resumen, "plantilla": PLANTILLA,
                      "filas": muestra[["mensaje", "motivo", "dificultad", "preentrenado", "bolsa"]].to_dict("records")}
muestra[["mensaje", "motivo", "dificultad", "preentrenado", "bolsa"]]

## 6 · Los cuatro casos en todos los modelos

**Predice antes de ejecutar:** para cada caso, ¿qué motivo responderá la bolsa de palabras? ¿Y el modelo preentrenado?

In [ ]:
tabla, probs = [], {}
zs_casos = zero_shot(CASOS["mensaje"], MOTIVOS)
for nombre in MODELOS:
    probs[nombre] = probabilidades(nombre, CASOS["mensaje"])
for i, c in CASOS.reset_index(drop=True).iterrows():
    fila = {"caso": c["id"], "referencia": c["referencia"]}
    for nombre in MODELOS:
        fila[nombre] = MOTIVOS[int(probs[nombre][i].argmax())]
    fila["Preentrenado (zero-shot)"] = max(zs_casos[i], key=zs_casos[i].get)
    tabla.append(fila)
SALIDAS["casos"] = [{**f, "mensaje": CASOS.iloc[i]["mensaje"],
                     "puntajes": {**{n: {m: float(p) for m, p in zip(MOTIVOS, probs[n][i])} for n in MODELOS},
                                  "Preentrenado (zero-shot)": zs_casos[i]}} for i, f in enumerate(tabla)]
pd.DataFrame(tabla).set_index("caso")

### ¿Por qué acierta la bolsa?

Estas son las palabras de A y de B que la bolsa conoce, con lo que cada una empuja hacia cada motivo según lo que aprendió. Una palabra que no está en su vocabulario **no cuenta para nada**. Acertar no siempre quiere decir haber entendido.

In [ ]:
vec_bolsa = MODELOS["Bolsa de palabras"].layers[0]
pesos = MODELOS["Bolsa de palabras"].layers[-1].get_weights()[0]       # una fila por palabra, una columna por motivo
vocab_bolsa = vec_bolsa.get_vocabulary()
SALIDAS["aporte"] = {}
for i in (0, 1):
    c = CASOS.iloc[i]
    palabras = re.findall(r"\w+", c["mensaje"].lower())
    fuera = [w for w in palabras if w not in vocab_bolsa]
    aporte = pd.DataFrame({w: pesos[vocab_bolsa.index(w)] for w in palabras if w in vocab_bolsa}, index=MOTIVOS).T.round(2)
    print(f'{c["id"]} · {c["mensaje"]}')
    print("   no están en el vocabulario, así que no cuentan:", fuera or "ninguna")
    display(aporte)
    SALIDAS["aporte"][c["id"]] = {"fuera": fuera, "pesos": {w: {m: float(v) for m, v in f.items()} for w, f in aporte.iterrows()}}

## 7 · Tu proyecto

Si todavía no tienes tema, elige uno de los dos conjuntos del laboratorio o trae el tuyo: un CSV con una columna de texto, una de etiqueta y, si puedes, una que agrupe los casos que se parecen demasiado (mismo cliente, misma plantilla, misma persona).

| Conjunto | Texto | Etiqueta | Agrupar por |
|---|---|---|---|
| `mensajes_soporte.csv` | `mensaje` | `motivo` (4) | `frase` |
| `opiniones.csv` | `opinion` | `sentimiento` (2) | `frase` |

Para usar tu CSV: súbelo con el ícono de carpeta de Colab y escribe su nombre en `datos`.

**Edita solo la celda «Cambia esto»** y ejecuta las siguientes. Al final se genera `mi_proyecto_e03.md`: guárdalo, lo vas a usar en el Encuentro 4.

In [ ]:
# ── Cambia esto ─────────────────────────────────────────────
PROYECTO = {
    "datos": "mensajes_soporte.csv",        # "opiniones.csv" o el nombre de tu CSV
    "columna_texto": "mensaje",             # en opiniones.csv: "opinion"
    "columna_etiqueta": "motivo",           # en opiniones.csv: "sentimiento"
    "columna_grupo": "frase",               # si tu CSV no la tiene, déjala así: se parte por fila
    "tarea": "Sugerir el motivo de cada mensaje para que una persona lo revise",
    "usuario": "El equipo que atiende el soporte",
    "error_que_importa": "Mandar un reclamo de cobro a la fila de envíos",
    "minimo": "Superar a la línea base también en los mensajes con negación",
    "mi_frase": "No me llegó la factura del pedido que pagué ayer.",
    "ngramas": 2,                           # 1 = palabras sueltas · 2 = también pares de palabras
}
# ────────────────────────────────────────────────────────────

In [ ]:
p = PROYECTO
fuente = p["datos"] if os.path.exists(p["datos"]) or "://" in p["datos"] else DATOS + p["datos"]
mi = pd.read_csv(fuente).dropna(subset=[p["columna_texto"], p["columna_etiqueta"]]).reset_index(drop=True)
ETIQUETAS = sorted(mi[p["columna_etiqueta"]].astype(str).unique())
mi["y"] = mi[p["columna_etiqueta"]].astype(str).map(ETIQUETAS.index)
grupos = mi[p["columna_grupo"]] if p["columna_grupo"] in mi.columns else mi.index
a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0).split(mi, groups=grupos))
mi_ent, mi_pru = mi.iloc[a], mi.iloc[b]

# 1) un modelo que ya existe, sin entrenar nada
zs_frase = zero_shot([p["mi_frase"]], ETIQUETAS)[0]
# 2) línea base y una bolsa de palabras entrenada con tus datos
base = mi_ent["y"].mode()[0]
mi_modelo = entrenar(mi_ent[p["columna_texto"]], mi_ent["y"],
                     layers.TextVectorization(max_tokens=10000, ngrams=p["ngramas"], output_mode="count"), clases=len(ETIQUETAS))
mi_pred = mi_modelo.predict(como_texto(mi_pru[p["columna_texto"]]), verbose=0).argmax(axis=1)
bolsa_frase = mi_modelo.predict(como_texto([p["mi_frase"]]), verbose=0)[0]
# 3) el modelo existente sobre una parte de tu prueba (máximo 40 textos, para no esperar)
parte = mi_pru.sample(min(40, len(mi_pru)), random_state=0)
zs_parte = [max(z, key=z.get) for z in zero_shot(parte[p["columna_texto"]], ETIQUETAS)]

exact = {"Línea base": float((mi_pru["y"] == base).mean()),
         "Bolsa entrenada": float((mi_pred == mi_pru["y"].values).mean()),
         f"Modelo existente ({len(parte)} textos)": float(np.mean([z == e for z, e in zip(zs_parte, parte[p["columna_etiqueta"]].astype(str))]))}
print(f"{len(mi)} textos · {len(mi_ent)} para entrenar · {len(mi_pru)} para probar · etiquetas: {ETIQUETAS}\n")
for k, v in exact.items():
    print(f"{k:>32}: {v:.3f}")
print(f"\nTu frase: «{p['mi_frase']}»")
print("   modelo existente :", max(zs_frase, key=zs_frase.get), {k: round(v, 2) for k, v in zs_frase.items()})
print("   bolsa entrenada  :", ETIQUETAS[int(bolsa_frase.argmax())], {e: round(float(v), 2) for e, v in zip(ETIQUETAS, bolsa_frase)})
SALIDAS["proyecto"] = {"datos": p["datos"], "etiquetas": ETIQUETAS, "textos": len(mi), "prueba": len(mi_pru),
                       "exactitud": exact, "mi_frase": p["mi_frase"], "existente": max(zs_frase, key=zs_frase.get),
                       "bolsa": ETIQUETAS[int(bolsa_frase.argmax())]}

### Tu ficha del día

Se genera sola con lo que ejecutaste. Complétala con tu caso difícil: un texto en el que la preparación o la representación cambió la respuesta.

In [ ]:
filas = "\n".join(f"| {k} | {v:.3f} |" for k, v in exact.items())
ficha = f'''# Mi proyecto · ficha del Encuentro 3

## Resumen
- **Tarea:** {p["tarea"]}
- **Quién usa la predicción:** {p["usuario"]}
- **Error que más importa:** {p["error_que_importa"]}
- **Mínimo aceptable:** {p["minimo"]}

## Datos
- Archivo: `{p["datos"]}` · {len(mi)} textos · etiquetas: {", ".join(ETIQUETAS)}
- Partición: 70 % entrenamiento / 30 % prueba, agrupando por `{p["columna_grupo"] if p["columna_grupo"] in mi.columns else "fila (sin columna de grupo)"}`

## Representación
- Preparación: minúsculas y sin signos de puntuación (`TextVectorization`). Palabras vacías: no se quitan.
- Representación: bolsa de palabras con ngramas = {p["ngramas"]}
- Modelo existente: `{MODELO}` (revisión `{REVISION[:10]}`), zero-shot con la plantilla «{PLANTILLA}»

## Primera predicción
- Frase: «{p["mi_frase"]}»
- Modelo existente: **{max(zs_frase, key=zs_frase.get)}**
- Bolsa entrenada: **{ETIQUETAS[int(bolsa_frase.argmax())]}**

## Resultados en la prueba
| Modelo | Exactitud |
|---|---|
{filas}

## Caso difícil (complétalo)
- Texto:
- Qué cambió y con qué representación:
- Qué no puedo concluir todavía:
'''
with open("mi_proyecto_e03.md", "w", encoding="utf-8") as f:
    f.write(ficha)
print(ficha)
import sys
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("mi_proyecto_e03.md")

## Guardar las salidas

`e03_salidas.json` conserva lo que se ejecutó, con fecha y versiones, para poder citarlo como evidencia.

In [ ]:
SALIDAS["segundos"] = round(time.time() - INICIO, 1)
with open("e03_salidas.json", "w", encoding="utf-8") as f:
    json.dump(SALIDAS, f, ensure_ascii=False, indent=2)
print(f"Guardado e03_salidas.json · {ENTORNO['fecha']} · {SALIDAS['segundos']} s")

## Para el próximo encuentro (sin tarea adicional)

Guarda `mi_proyecto_e03.md`. En el Encuentro 4 vas a medir con más cuidado: qué pasa si los datos se parten mal, qué errores comete cada modelo por categoría y cuáles de esos errores justificarían adaptar un modelo.

Módulo relacionado: **M2 · Procesamiento de lenguaje natural** (semana 2).